# Rope Flow — Full Pipeline V04
## Data-Driven Pattern Classification (7-class)

**Course:** MECH 798M / EECE 798K — Data-Driven Modeling  
**Input:** Processed CSV pairs (`_device0_processed.csv` + `_device1_processed.csv`)  
**Output:** Per-cycle + session-level pattern classification with transition rejection

### Pipeline architecture (56D feature vector)

| # | Stage | Dim | DDM Pillar |
|---|-------|-----|-----------|
| 1 | Cycle detection | — | Signal processing |
| 2 | Physics features | 12D | Pillar 6: Scientific modeling |
| 3 | Fourier decomposition | 16D | Pillar 5: Time series |
| 4 | SVD features | 8D | Pillar 4: Dimensionality reduction |
| 5 | Template correlation | 2D | Template matching |
| 6 | DMD eigenvalues | 9D | Pillar 4: Dimensionality reduction |
| 7 | Subject-invariant | 9D | Pillar 6: Scientific modeling |

---
## 0. Setup & configuration

In [9]:
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from scipy.linalg import svd
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────
DATA_PROCESSED = os.path.join('..', '..', 'data', 'processed')

CONFIG = {
    'FS': 50.0,
    'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0,
    'TARGET_LEN': 64,
    'N_FOURIER_HARMONICS': 5,
    'PCA_COMPONENTS': 6,
    'TEMPLATE_CORR_THRESHOLD': 0.60,
    'OCSVM_NU': 0.1,
    'OCSVM_GAMMA': 'scale',
    'MIN_CYCLE_SAMPLES': 10,
    'CONFIDENCE_THRESHOLD': 0.7,      # GBM confidence gate for transition rejection
    'TEMPLATE_GATE_THRESHOLD': 0.2,   # min template corr to accept a cycle
}

MACRO_MAP = {
    'underhand': 'foundational', 'overhand': 'foundational',
    'dragon_roll': 'advanced',
    'sneak_underhand': 'sneak', 'sneak_overhand': 'sneak',
    'race_and_chase': 'advanced', 'cheetahs_tail': 'advanced',
}

KNOWN_PATTERNS = set(MACRO_MAP.keys())

PAL = {
    'overhand': '#7F77DD', 'underhand': '#5DCAA5', 'dragon_roll': '#E24B4A',
    'sneak_overhand': '#D85A30', 'sneak_underhand': '#EF9F27',
    'race_and_chase': '#3498db', 'cheetahs_tail': '#e74c3c',
    'transition': '#CCCCCC', 'unknown': '#888888',
}

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

---
## 1. Core pipeline functions (V02 — validated)

All feature extraction, template, and base classification functions.

In [10]:
def load_session(path_d0, path_d1):
    d0 = pd.read_csv(path_d0)
    d1 = pd.read_csv(path_d1)
    return d0, d1

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w', 'ay_w', 'az_w']].values
    omega = df[['gx', 'gy', 'gz']].values * (np.pi / 180.0)
    Q = df[['qw', 'qx', 'qy', 'qz']].values
    return t, Q, A, omega

# ══════════════════════════════════════════════════════════════
# 2. Cycle detection
# ══════════════════════════════════════════════════════════════
def detect_cycles(t, omega, fs=50.0):
    mag = np.linalg.norm(omega, axis=1)
    mag_smooth = savgol_filter(mag, window_length=15, polyorder=3)
    prom_rads = CONFIG['CYCLE_PROMINENCE_DEGS'] * np.pi / 180.0
    min_dist = int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)
    peaks, _ = find_peaks(mag_smooth, distance=min_dist, prominence=prom_rads)
    if len(peaks) < 2:
        return [], mag_smooth, peaks
    bounds = ([0]
              + [(peaks[i] + peaks[i+1]) // 2 for i in range(len(peaks) - 1)]
              + [len(t) - 1])
    cycles = [(bounds[i], bounds[i+1]) for i in range(len(bounds) - 1)]
    valid = []
    for s, e in cycles:
        dur = t[e] - t[s]
        if (dur >= CONFIG['CYCLE_MIN_PERIOD_S'] and
            dur <= CONFIG['CYCLE_MAX_PERIOD_S'] and
            (e - s) >= CONFIG['MIN_CYCLE_SAMPLES']):
            valid.append((s, e))
    return valid, mag_smooth, peaks

def pair_cycles(t0, cyc0, t1, cyc1):
    def overlap(s0, e0, s1, e1):
        return max(0.0, min(t0[e0], t1[e1]) - max(t0[s0], t1[s1]))
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_idx, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used:
                continue
            ov = overlap(c0[0], c0[1], c1[0], c1[1])
            if ov > best_ov:
                best_ov, best_idx = ov, i
        if best_idx >= 0 and best_ov > 0:
            paired0.append(c0)
            paired1.append(cyc1[best_idx])
            used.add(best_idx)
    return paired0, paired1

# ══════════════════════════════════════════════════════════════
# 3. Fourier decomposition
# ══════════════════════════════════════════════════════════════
def fourier_features(signal_segment, fs, n_harmonics=5):
    N = len(signal_segment)
    if N < 8:
        return np.zeros(3 + n_harmonics)
    sig = signal_segment - np.mean(signal_segment)
    F = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(N, 1.0 / fs)
    F[0] = 0
    power = F ** 2
    total_power = np.sum(power) + 1e-12
    dom_idx = np.argmax(F[1:]) + 1
    dom_freq = freqs[dom_idx]
    spectral_centroid = np.sum(freqs * power) / total_power
    f0 = dom_freq
    harmonic_ratios = np.zeros(n_harmonics)
    for h in range(n_harmonics):
        target_freq = f0 * (h + 1)
        if target_freq > freqs[-1]:
            break
        idx = np.argmin(np.abs(freqs - target_freq))
        lo = max(1, idx - 1)
        hi = min(len(F), idx + 2)
        harmonic_ratios[h] = np.sum(power[lo:hi]) / total_power
    p_norm = power[1:] / (np.sum(power[1:]) + 1e-12)
    p_norm = p_norm[p_norm > 0]
    spectral_entropy = -np.sum(p_norm * np.log2(p_norm + 1e-12))
    return np.concatenate([[dom_freq, spectral_centroid, spectral_entropy],
                           harmonic_ratios])

def cycle_fourier_features(A, omega, s, e, fs):
    nh = CONFIG['N_FOURIER_HARMONICS']
    feats = []
    omega_mag = np.linalg.norm(omega[s:e], axis=1)
    feats.append(fourier_features(omega_mag, fs, nh))
    acc_mag = np.linalg.norm(A[s:e], axis=1)
    feats.append(fourier_features(acc_mag, fs, nh))
    return np.concatenate(feats)

FOURIER_DIM = 2 * (3 + CONFIG['N_FOURIER_HARMONICS'])
FOURIER_NAMES = []
for ch in ['omega_mag', 'acc_mag']:
    FOURIER_NAMES += [f'{ch}_dom_freq', f'{ch}_spec_centroid', f'{ch}_spec_entropy']
    FOURIER_NAMES += [f'{ch}_harm_{i}' for i in range(CONFIG['N_FOURIER_HARMONICS'])]

# ══════════════════════════════════════════════════════════════
# 4. Physics-grounded features
# ══════════════════════════════════════════════════════════════
def physics_features(t0, t1, A0, A1, om0, om1, s0, e0, s1, e1, fs):
    tc0, tc1 = t0[s0:e0], t1[s1:e1]
    a0, a1 = A0[s0:e0], A1[s1:e1]
    w0, w1 = om0[s0:e0], om1[s1:e1]
    period = tc0[-1] - tc0[0]
    m0 = np.linalg.norm(w0, axis=1)
    m1 = np.linalg.norm(w1, axis=1)
    peak_omega = max(np.max(m0), np.max(m1))
    mean_omega_d0 = np.mean(m0)
    mean_omega_d1 = np.mean(m1)
    acc_rms_d0 = np.sqrt(np.mean(np.sum(a0**2, axis=1)))
    acc_rms_d1 = np.sqrt(np.mean(np.sum(a1**2, axis=1)))
    acc_asym = acc_rms_d0 / (acc_rms_d1 + 1e-8)
    omega_ratio = np.std(m0) / (np.std(m1) + 1e-8)
    n_min = min(len(m0), len(m1))
    if n_min > 4:
        c0 = m0[:n_min] - m0[:n_min].mean()
        c1 = m1[:n_min] - m1[:n_min].mean()
        corr = np.correlate(c0, c1, mode='full')
        lag_samples = np.argmax(corr) - (n_min - 1)
        phase_lag = lag_samples / fs * 1000
    else:
        phase_lag = 0.0
    if len(a0) > 2:
        jerk = np.diff(a0, axis=0) * fs
        jerk_rms = np.sqrt(np.mean(np.sum(jerk**2, axis=1)))
    else:
        jerk_rms = 0.0
    e_rot = np.mean(m0**2) + np.mean(m1**2)
    e_lin = np.mean(np.sum(a0**2, axis=1)) + np.mean(np.sum(a1**2, axis=1))
    ke_ratio = e_rot / (e_rot + e_lin + 1e-8)
    if len(m0) > 10:
        m0c = m0 - m0.mean()
        ac = np.correlate(m0c, m0c, mode='full')
        ac = ac[len(m0c)-1:]
        ac /= (ac[0] + 1e-12)
        min_lag = max(2, int(0.3 * fs))
        peaks_ac, props = find_peaks(ac[min_lag:], height=0)
        periodicity = props['peak_heights'][0] if len(peaks_ac) > 0 else 0.0
    else:
        periodicity = 0.0
    return np.array([
        period, peak_omega, mean_omega_d0, mean_omega_d1,
        acc_rms_d0, acc_rms_d1, acc_asym, omega_ratio,
        phase_lag, jerk_rms, ke_ratio, periodicity
    ])

PHYSICS_NAMES = [
    'period_s', 'peak_omega', 'mean_omega_d0', 'mean_omega_d1',
    'acc_rms_d0', 'acc_rms_d1', 'acc_asymmetry', 'omega_std_ratio',
    'phase_lag_ms', 'jerk_rms', 'ke_ratio', 'periodicity_score'
]

# ══════════════════════════════════════════════════════════════
# 5. SVD features
# ══════════════════════════════════════════════════════════════
def resample_cycle(signal, target_len):
    n = len(signal)
    if n < 2:
        return np.zeros(target_len) if signal.ndim == 1 else np.zeros((target_len, signal.shape[1]))
    x_old = np.linspace(0, 1, n)
    x_new = np.linspace(0, 1, target_len)
    if signal.ndim == 1:
        return np.interp(x_new, x_old, signal)
    else:
        return np.column_stack([np.interp(x_new, x_old, signal[:, j])
                                for j in range(signal.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1, target_len=64):
    state0 = np.column_stack([A0[s0:e0], om0[s0:e0]])
    state1 = np.column_stack([A1[s1:e1], om1[s1:e1]])
    r0 = resample_cycle(state0, target_len)
    r1 = resample_cycle(state1, target_len)
    return np.column_stack([r0, r1]).T  # (12, target_len)

def compute_svd_features(cycle_matrix, n_components=6):
    U, S, Vt = svd(cycle_matrix, full_matrices=False)
    S_ratio = S[:n_components] / (S[0] + 1e-12)
    eff_rank = np.sum(S > 0.01 * S[0])
    total_energy = np.sum(S**2)
    return np.concatenate([S_ratio, [eff_rank, total_energy]])

SVD_DIM = CONFIG['PCA_COMPONENTS'] + 2
SVD_NAMES = ([f'sv_ratio_{i}' for i in range(CONFIG['PCA_COMPONENTS'])]
             + ['eff_rank', 'total_energy'])

# ══════════════════════════════════════════════════════════════
# 6. Template correlation
# ══════════════════════════════════════════════════════════════
def phase_align(cycle_matrix, reference, ref_channel=3):
    """
    Circularly shift cycle_matrix to maximize correlation with reference
    on a chosen channel (default: ωx of D0, row 3).
    This removes arbitrary phase offset from cycle segmentation.
    """
    x = cycle_matrix[ref_channel]
    y = reference[ref_channel]
    x_c = x - np.mean(x)
    y_c = y - np.mean(y)
    corr = np.correlate(np.tile(x_c, 2), y_c, mode='valid')
    shift = np.argmax(corr)
    if shift >= len(x):
        shift -= len(x)
    aligned = np.roll(cycle_matrix, -shift, axis=1)
    return aligned, shift


def build_template(all_cycle_matrices):
    """
    Build overhand template via iterative phase alignment.
    1. Initial template = median-energy cycle
    2. Phase-align all cycles to it
    3. Average the IQR-filtered aligned cycles
    """
    energies = np.array([np.sum(m**2) for m in all_cycle_matrices])
    # Start from the cycle closest to median energy
    median_idx = np.argmin(np.abs(energies - np.median(energies)))
    template = all_cycle_matrices[median_idx].copy()

    # Iterative refinement (2 passes)
    for iteration in range(2):
        aligned = []
        for cm in all_cycle_matrices:
            cm_a, _ = phase_align(cm, template)
            aligned.append(cm_a)
        # IQR filter on energy
        q25, q75 = np.percentile(energies, [25, 75])
        mask = (energies >= q25) & (energies <= q75)
        selected = [a for a, keep in zip(aligned, mask) if keep]
        if len(selected) < 3:
            selected = aligned
        template = np.mean(selected, axis=0)

    return template


def template_correlation(cycle_matrix, template):
    """
    Phase-aligned normalized correlation between cycle and template.
    Returns mean and min correlation across all 12 channels.
    """
    aligned, _ = phase_align(cycle_matrix, template)
    n_channels = aligned.shape[0]
    corrs = np.zeros(n_channels)
    for ch in range(n_channels):
        x = aligned[ch] - np.mean(aligned[ch])
        y = template[ch] - np.mean(template[ch])
        denom = (np.linalg.norm(x) * np.linalg.norm(y) + 1e-12)
        corrs[ch] = np.dot(x, y) / denom
    return np.mean(corrs), np.min(corrs)

TEMPLATE_NAMES = ['template_corr_mean', 'template_corr_min']

# ══════════════════════════════════════════════════════════════
# 7. DMD eigenvalue features — dynamical fingerprint
# ══════════════════════════════════════════════════════════════
def dmd_features(cycle_matrix, dt):
    """
    DMD eigenvalue features — subject-invariant dynamical fingerprint.
    Captures the linear operator governing one cycle's dynamics.
    Returns top-3 frequencies, top-3 growth rates, rank, mean/max stability.
    """
    X1, X2 = cycle_matrix[:, :-1], cycle_matrix[:, 1:]
    U, S, Vh = svd(X1, full_matrices=False)
    cumvar = np.cumsum(S**2) / (np.sum(S**2) + 1e-12)
    r = min(int(np.searchsorted(cumvar, 0.99)) + 1, len(S), X1.shape[1])
    r = max(r, 1)
    A_t = U[:,:r].T @ X2 @ Vh[:r,:].T @ np.diag(1.0 / (S[:r] + 1e-12))
    lam_d, _ = np.linalg.eig(A_t)
    lam_c = np.log(np.maximum(np.abs(lam_d), 1e-12)) / dt
    freqs = np.abs(lam_c.imag) / (2 * np.pi)
    growth = lam_c.real
    order = np.argsort(freqs)[::-1]
    freqs, growth = freqs[order], growth[order]
    n_top = 3
    f_top, g_top = np.zeros(n_top), np.zeros(n_top)
    for i in range(min(n_top, len(freqs))):
        f_top[i] = freqs[i]; g_top[i] = growth[i]
    return np.concatenate([f_top, g_top, [r, np.mean(growth), np.max(np.abs(growth))]])

DMD_FEAT_NAMES = ([f'dmd_freq_{i}' for i in range(3)]
                  + [f'dmd_growth_{i}' for i in range(3)]
                  + ['dmd_rank', 'dmd_mean_growth', 'dmd_max_abs_growth'])


# ══════════════════════════════════════════════════════════════
# 8. Subject-invariant features — ratios that cancel person effects
# ══════════════════════════════════════════════════════════════
def subject_invariant_features(t0, t1, A0, A1, om0, om1, s0, e0, s1, e1, fs):
    """
    Features designed to be invariant to subject-specific scaling.
    Uses D1/D0 ratios, statistical shape (skewness, kurtosis),
    and inter-hand correlation — all dimensionless.
    """
    from scipy.stats import skew, kurtosis as kurt
    a0, a1 = A0[s0:e0], A1[s1:e1]
    w0, w1 = om0[s0:e0], om1[s1:e1]
    m0 = np.linalg.norm(w0, axis=1)
    m1 = np.linalg.norm(w1, axis=1)
    am0 = np.linalg.norm(a0, axis=1)
    am1 = np.linalg.norm(a1, axis=1)
    n = min(len(m0), len(m1))
    return np.array([
        np.mean(m1) / (np.mean(m0) + 1e-8),       # ω ratio D1/D0
        np.mean(am1) / (np.mean(am0) + 1e-8),      # acc ratio D1/D0
        np.std(m1) / (np.std(m0) + 1e-8),           # ω variability ratio
        np.max(m1) / (np.max(m0) + 1e-8),           # peak ω ratio
        skew(m0), skew(m1),                          # temporal shape
        kurt(m0), kurt(m1),
        np.corrcoef(m0[:n], m1[:n])[0, 1] if n > 4 else 0.0,  # inter-hand coupling
    ])

SI_FEAT_NAMES = ['omega_ratio_d1d0', 'acc_ratio_d1d0', 'omega_std_ratio_si',
                 'peak_omega_ratio', 'skew_omega_d0', 'skew_omega_d1',
                 'kurt_omega_d0', 'kurt_omega_d1', 'phase_coupling']


# ══════════════════════════════════════════════════════════════
# 9. Full feature extraction
# ══════════════════════════════════════════════════════════════
ALL_FEAT_NAMES = (PHYSICS_NAMES + FOURIER_NAMES + SVD_NAMES + TEMPLATE_NAMES
                  + DMD_FEAT_NAMES + SI_FEAT_NAMES)
N_FEAT = len(ALL_FEAT_NAMES)

def extract_all_features(t0, t1, A0, A1, om0, om1,
                         paired0, paired1, template, fs):
    features = []
    dt = 1.0 / fs
    for (s0, e0), (s1, e1) in zip(paired0, paired1):
        phys = physics_features(t0, t1, A0, A1, om0, om1, s0, e0, s1, e1, fs)
        fourier = cycle_fourier_features(A0, om0, s0, e0, fs)
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1,
                                CONFIG['TARGET_LEN'])
        svd_feat = compute_svd_features(cm, CONFIG['PCA_COMPONENTS'])
        corr_mean, corr_min = template_correlation(cm, template)
        tmpl_feat = np.array([corr_mean, corr_min])
        dmd_feat = dmd_features(cm, dt)
        si_feat = subject_invariant_features(t0, t1, A0, A1, om0, om1,
                                             s0, e0, s1, e1, fs)
        feat = np.concatenate([phys, fourier, svd_feat, tmpl_feat,
                               dmd_feat, si_feat])
        features.append(feat)
    return np.array(features)

# ══════════════════════════════════════════════════════════════
# 8. Classification
# ══════════════════════════════════════════════════════════════
def train_overhand_detector(X_features):
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_features)
    ocsvm = OneClassSVM(
        kernel='rbf',
        nu=CONFIG['OCSVM_NU'],
        gamma=CONFIG['OCSVM_GAMMA']
    )
    ocsvm.fit(X_s)
    stats = {
        'mean': np.mean(X_features, axis=0),
        'std': np.std(X_features, axis=0),
        'q25': np.percentile(X_features, 25, axis=0),
        'q75': np.percentile(X_features, 75, axis=0),
    }
    return ocsvm, scaler, stats

def classify_cycles(X_features, ocsvm, scaler, template_corr_idx):
    """
    Classify each cycle as 'overhand' or 'unknown'.

    Two-stage approach:
    1. Per-cycle: combined score from SVM decision + template correlation
    2. Session-level: if >50% of cycles are overhand, boost borderline cycles

    Confidence = weighted combination of:
      - One-class SVM decision function (learned distribution)
      - Template correlation (physics-based similarity)
    """
    X_s = scaler.transform(X_features)
    svm_decision = ocsvm.decision_function(X_s)

    corr_mean = X_features[:, template_corr_idx]

    # Normalize SVM decision to [0, 1]
    svm_min, svm_max = np.min(svm_decision), np.max(svm_decision)
    svm_norm = (svm_decision - svm_min) / (svm_max - svm_min + 1e-12)

    # Normalize template correlation to [0, 1]
    corr_min_val, corr_max_val = np.min(corr_mean), np.max(corr_mean)
    corr_norm = (corr_mean - corr_min_val) / (corr_max_val - corr_min_val + 1e-12)

    # Combined confidence: 60% SVM + 40% template
    confidence = 0.6 * svm_norm + 0.4 * corr_norm

    # Stage 1: strict threshold — bottom 5% are definitely outliers
    strict_threshold = np.percentile(confidence, 5)
    initial_labels = ['overhand' if c > strict_threshold else 'unknown'
                      for c in confidence]

    # Stage 2: session-level vote — if majority is overhand, classify session
    n_oh = sum(1 for l in initial_labels if l == 'overhand')
    session_is_overhand = (n_oh / len(initial_labels)) > 0.5

    if session_is_overhand:
        # Relax threshold: only reject extreme outliers (bottom 2%)
        relax_threshold = np.percentile(confidence, 2)
        labels = ['overhand' if c > relax_threshold else 'unknown'
                  for c in confidence]
    else:
        labels = initial_labels

    return labels, confidence.tolist()

# ══════════════════════════════════════════════════════════════
# 9. Visualization
# ══════════════════════════════════════════════════════════════
def plot_pipeline_summary(t0, omega0_mag_smooth, peaks0, paired0,
                          X_features, labels, confidences, template,
                          save_path=None):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Panel 1: Cycle detection
    ax = axes[0, 0]
    t_plot = np.arange(len(omega0_mag_smooth)) / CONFIG['FS']
    ax.plot(t_plot, omega0_mag_smooth, 'k-', lw=0.5, alpha=0.7)
    valid_peaks = [p for p in peaks0 if p < len(t_plot)]
    ax.plot([t_plot[p] for p in valid_peaks],
            [omega0_mag_smooth[p] for p in valid_peaks],
            'rv', ms=3, alpha=0.6)
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('||ω|| [rad/s]')
    ax.set_title(f'Cycle detection: {len(paired0)} paired cycles')

    # Panel 2: Template ω channels
    ax = axes[0, 1]
    x = np.linspace(0, 1, template.shape[1])
    for ch, color, lbl in [(3, '#5DCAA5', 'ωx'), (4, '#7F77DD', 'ωy'), (5, '#E24B4A', 'ωz')]:
        ax.plot(x, template[ch], color=color, lw=2, label=f'template {lbl}')
    ax.set_xlabel('Normalized cycle phase')
    ax.set_ylabel('Angular velocity [rad/s]')
    ax.set_title('Overhand template (D0 ω channels)')
    ax.legend(fontsize=8)

    # Panel 3: PCA projection
    ax = axes[1, 0]
    if len(X_features) > 3:
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(StandardScaler().fit_transform(X_features))
        colors = ['#5DCAA5' if l == 'overhand' else '#E24B4A' for l in labels]
        ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=20)
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('Feature space (PCA projection)')

    # Panel 4: Confidence timeline
    ax = axes[1, 1]
    colors = ['#5DCAA5' if l == 'overhand' else '#E24B4A' for l in labels]
    n_plot = min(len(paired0), len(confidences))
    ax.bar(range(n_plot), confidences[:n_plot], color=colors[:n_plot], alpha=0.7)
    ax.axhline(CONFIG['TEMPLATE_CORR_THRESHOLD'], color='gray',
               ls='--', lw=1, label=f'threshold={CONFIG["TEMPLATE_CORR_THRESHOLD"]}')
    ax.set_xlabel('Cycle index')
    ax.set_ylabel('Confidence')
    ax.set_title('Classification confidence per cycle')
    ax.legend(fontsize=8)

    plt.suptitle('Rope Flow Pipeline V02 — Overhand Detection', fontsize=14, y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    return fig

# ══════════════════════════════════════════════════════════════
# 10. Main pipeline
# ══════════════════════════════════════════════════════════════
def run_pipeline(path_d0, path_d1, pattern_label='overhand', verbose=True):
    fs = CONFIG['FS']
    d0, d1 = load_session(path_d0, path_d1)
    t0, Q0, A0, om0 = extract_signals(d0)
    t1, Q1, A1, om1 = extract_signals(d1)
    if verbose:
        print(f'Loaded: {len(d0)} samples, {t0[-1]:.1f} s duration, {fs:.0f} Hz')

    cyc0, mag0_smooth, peaks0 = detect_cycles(t0, om0, fs)
    cyc1, mag1_smooth, peaks1 = detect_cycles(t1, om1, fs)
    paired0, paired1 = pair_cycles(t0, cyc0, t1, cyc1)
    if verbose:
        print(f'Cycles: D0={len(cyc0)}, D1={len(cyc1)}, paired={len(paired0)}')
    if len(paired0) < 3:
        print('ERROR: Too few cycles detected.')
        return None

    cycle_matrices = []
    for (s0, e0), (s1, e1) in zip(paired0, paired1):
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1,
                                CONFIG['TARGET_LEN'])
        cycle_matrices.append(cm)

    template = build_template(cycle_matrices)
    if verbose:
        print(f'Template built from {len(cycle_matrices)} cycles '
              f'({CONFIG["TARGET_LEN"]} samples x 12 channels)')

    X = extract_all_features(t0, t1, A0, A1, om0, om1,
                             paired0, paired1, template, fs)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    if verbose:
        print(f'Feature matrix: {X.shape[0]} cycles x {X.shape[1]} features')

    ocsvm, scaler, stats = train_overhand_detector(X)

    template_corr_idx = ALL_FEAT_NAMES.index('template_corr_mean')
    labels, confidences = classify_cycles(X, ocsvm, scaler, template_corr_idx)

    n_overhand = sum(1 for l in labels if l == 'overhand')
    accuracy = n_overhand / len(labels) * 100
    if verbose:
        print(f'\n{"="*60}')
        print(f'CLASSIFICATION RESULT')
        print(f'{"="*60}')
        print(f'  Ground truth: {pattern_label}')
        print(f'  Total cycles: {len(labels)}')
        print(f'  Overhand:     {n_overhand} ({accuracy:.1f}%)')
        print(f'  Unknown:      {len(labels) - n_overhand} ({100-accuracy:.1f}%)')
        print(f'  Mean confidence: {np.mean(confidences):.3f}')
        print(f'{"="*60}')

    return {
        'X': X, 'labels': labels, 'confidences': confidences,
        'template': template, 'ocsvm': ocsvm, 'scaler': scaler,
        'stats': stats, 'paired0': paired0, 'paired1': paired1,
        'cycle_matrices': cycle_matrices, 'feature_names': ALL_FEAT_NAMES,
        't0': t0, 't1': t1, 'A0': A0, 'A1': A1, 'om0': om0, 'om1': om1,
        'mag0_smooth': mag0_smooth, 'peaks0': peaks0,
        'accuracy': accuracy, 'pattern_label': pattern_label,
    }


# ══════════════════════════════════════════════════════════════
# 11. Multi-session training & cross-validation
# ══════════════════════════════════════════════════════════════
def process_session(path_d0, path_d1):
    """Load and preprocess a single session. Returns signals + cycle info."""
    d0, d1 = load_session(path_d0, path_d1)
    t0, Q0, A0, om0 = extract_signals(d0)
    t1, Q1, A1, om1 = extract_signals(d1)
    fs = CONFIG['FS']
    cyc0, mag0_smooth, peaks0 = detect_cycles(t0, om0, fs)
    cyc1, _, _ = detect_cycles(t1, om1, fs)
    p0, p1 = pair_cycles(t0, cyc0, t1, cyc1)
    cms = [build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1, CONFIG['TARGET_LEN'])
           for (s0, e0), (s1, e1) in zip(p0, p1)]
    return {
        't0': t0, 't1': t1, 'A0': A0, 'A1': A1, 'om0': om0, 'om1': om1,
        'paired0': p0, 'paired1': p1, 'cycle_matrices': cms,
        'mag0_smooth': mag0_smooth, 'peaks0': peaks0,
    }


def train_multi_session(session_paths_and_labels, verbose=True):
    """
    Train a multi-class pattern classifier from labeled sessions.

    Parameters
    ----------
    session_paths_and_labels : list of (d0_path, d1_path, label) tuples
    verbose : bool

    Returns
    -------
    model : dict with template, classifier, scaler, feature_names
    """
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.preprocessing import LabelEncoder

    fs = CONFIG['FS']
    all_cms, all_sessions, all_labels = [], [], []

    for d0_path, d1_path, label in session_paths_and_labels:
        sess = process_session(d0_path, d1_path)
        all_sessions.append(sess)
        all_cms.extend(sess['cycle_matrices'])
        all_labels.append(label)
        if verbose:
            print(f'  {label:<20s}: {len(sess["paired0"])} cycles')

    # Global template
    template = build_template(all_cms)

    # Extract features + per-session z-scoring
    X_list, y_list, sid_list = [], [], []
    for si, (sess, label) in enumerate(zip(all_sessions, all_labels)):
        X = extract_all_features(
            sess['t0'], sess['t1'], sess['A0'], sess['A1'],
            sess['om0'], sess['om1'], sess['paired0'], sess['paired1'],
            template, fs)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        X_list.append(X)
        y_list.extend([label] * len(X))
        sid_list.extend([si] * len(X))

    X_all = np.vstack(X_list)
    y_all = np.array(y_list)
    sid_all = np.array(sid_list)

    # Per-session z-scoring to remove subject effects
    X_z = np.zeros_like(X_all)
    for sid in np.unique(sid_all):
        mask = sid_all == sid
        mu = np.mean(X_all[mask], axis=0)
        sigma = np.std(X_all[mask], axis=0) + 1e-8
        X_z[mask] = (X_all[mask] - mu) / sigma

    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_z)

    le = LabelEncoder()
    y_enc = le.fit_transform(y_all)

    clf = GradientBoostingClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        random_state=42)
    clf.fit(X_s, y_enc)

    if verbose:
        print(f'Trained on {X_all.shape[0]} cycles × {X_all.shape[1]} features '
              f'from {len(session_paths_and_labels)} sessions')
        print(f'Classes: {list(le.classes_)}')

    return {
        'template': template,
        'clf': clf,
        'scaler': scaler,
        'label_encoder': le,
        'feature_names': ALL_FEAT_NAMES,
        'X_train': X_all,
        'X_train_z': X_z,
        'y_train': y_all,
        'session_ids': sid_all,
    }


def classify_session(path_d0, path_d1, model, verbose=True):
    """
    Classify a new session against a trained multi-class model.
    Uses per-session z-scoring + majority vote for session-level verdict.
    """
    fs = CONFIG['FS']
    sess = process_session(path_d0, path_d1)

    X = extract_all_features(
        sess['t0'], sess['t1'], sess['A0'], sess['A1'],
        sess['om0'], sess['om1'], sess['paired0'], sess['paired1'],
        model['template'], fs)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if X.ndim < 2 or len(X) == 0:
        if verbose:
            print(f'  0 valid cycles — skipping')
        return {
            'labels': [], 'confidences': [], 'X': np.empty((0, len(model['feature_names']))),
            'verdict': 'unknown', 'vote': {}, 'n_transition': 0, 'session': sess,
        }

    # Per-session z-scoring
    mu = np.mean(X, axis=0)

    if X.ndim < 2 or len(X) == 0:
        if verbose:
            print(f'  0 valid cycles — skipping')
        return {
            'labels': [], 'confidences': [], 'X': np.empty((0, len(model['feature_names']))),
            'verdict': 'unknown', 'vote': {}, 'n_transition': 0, 'session': sess,
        }

    sigma = np.std(X, axis=0) + 1e-8
    X_z = (X - mu) / sigma
    X_s = model['scaler'].transform(X_z)

    le = model['label_encoder']
    y_pred_enc = model['clf'].predict(X_s)
    y_pred_proba = model['clf'].predict_proba(X_s)
    labels = le.inverse_transform(y_pred_enc).tolist()

    # Confidence = max predicted probability
    confidences = np.max(y_pred_proba, axis=1).tolist()

    # Session-level vote
    from collections import Counter
    vote = Counter(labels)
    verdict = vote.most_common(1)[0][0]
    vote_pct = vote.most_common(1)[0][1] / len(labels) * 100

    if verbose:
        print(f'  Cycles: {len(labels)} | ', end='')
        for cls, cnt in sorted(vote.items()):
            print(f'{cls}={cnt} ({cnt/len(labels)*100:.0f}%) ', end='')
        print(f'→ {verdict}')

    return {
        'labels': labels,
        'confidences': confidences,
        'X': X,
        'verdict': verdict,
        'vote_pct': vote_pct,
        'vote': dict(vote),
        'session': sess,
    }



---
## 2. V04 functions: auto-discovery, transition rejection, LOSO

In [11]:
# ══════════════════════════════════════════════════════════════
# Session auto-discovery from processed CSV files
# ══════════════════════════════════════════════════════════════
def discover_sessions(processed_dir):
    """
    Auto-discover labeled sessions from processed CSV pairs.
    Filename format: YYYYMMDD_HHMMSS_patternname_subject_deviceN_processed.csv
    
    Returns list of (d0_path, d1_path, label, display_name) for homogeneous sessions.
    """
    csv_files = sorted(glob.glob(os.path.join(processed_dir, '*_device0_processed.csv')))
    sessions = []
    
    for d0_path in csv_files:
        d1_path = d0_path.replace('_device0_', '_device1_')
        if not os.path.exists(d1_path):
            continue
        
        basename = os.path.basename(d0_path)
        # Parse: YYYYMMDD_HHMMSS_patternname_subject_device0_processed.csv
        # Remove _device0_processed.csv suffix
        stem = basename.replace('_device0_processed.csv', '')
        parts = stem.split('_')
        
        if len(parts) < 3:
            continue
        
        # First two parts are date and time
        date_str, time_str = parts[0], parts[1]
        # Remaining parts (minus last = subject) form the pattern name
        # Subject is the last part
        remaining = parts[2:]
        
        # Try to match against known patterns (greedy: longest match first)
        label = None
        subject = None
        for n_parts in range(len(remaining), 0, -1):
            candidate = '_'.join(remaining[:n_parts])
            if candidate in KNOWN_PATTERNS:
                label = candidate
                subject = '_'.join(remaining[n_parts:]) if n_parts < len(remaining) else 'unknown'
                break
        
        if label is None:
            continue  # not a labeled session
        
        display_name = f'{label}_{subject}'
        sessions.append((d0_path, d1_path, label, display_name))
    
    return sessions


def summarize_sessions(sessions):
    """Print a summary of discovered sessions by class."""
    by_class = {}
    for d0, d1, label, name in sessions:
        by_class.setdefault(label, []).append(name)
    
    print(f'Discovered {len(sessions)} labeled sessions across {len(by_class)} classes:')
    for label in sorted(by_class.keys()):
        names = by_class[label]
        loso = 'LOSO OK' if len(names) >= 2 else 'SINGLETON'
        print(f'  {label:<20s}: {len(names)} sessions [{loso}]')
        for n in names:
            print(f'    - {n}')
    
    singletons = [l for l, s in by_class.items() if len(s) == 1]
    if singletons:
        print(f'\nWarning: {len(singletons)} singleton classes excluded from LOSO: {singletons}')
    
    return by_class


# ══════════════════════════════════════════════════════════════
# V04 classifier with transition rejection
# ══════════════════════════════════════════════════════════════
def train_model_v04(session_list, verbose=True):
    """
    Train multi-class GBM with per-class templates for transition rejection.
    
    Parameters
    ----------
    session_list : list of (d0_path, d1_path, label) tuples
    
    Returns
    -------
    model : dict with clf, scaler, template, class_templates, label_encoder, etc.
    """
    fs = CONFIG['FS']
    all_cms, all_sessions, all_labels = [], [], []
    
    for d0_path, d1_path, label in session_list:
        sess = process_session(d0_path, d1_path)
        all_sessions.append(sess)
        all_cms.extend(sess['cycle_matrices'])
        all_labels.append(label)
        if verbose:
            print(f'  {label:<20s}: {len(sess["paired0"]):>3d} cycles')
    
    # Global template
    template = build_template(all_cms)
    
    # Per-class templates for transition gating
    class_templates = {}
    offset = 0
    class_cms = {}
    for sess, label in zip(all_sessions, all_labels):
        n = len(sess['cycle_matrices'])
        class_cms.setdefault(label, []).extend(sess['cycle_matrices'])
        offset += n
    for label, cms in class_cms.items():
        class_templates[label] = build_template(cms)
    
    # Extract features + per-session z-scoring
    X_list, y_list, sid_list = [], [], []
    for si, (sess, label) in enumerate(zip(all_sessions, all_labels)):
        X = extract_all_features(
            sess['t0'], sess['t1'], sess['A0'], sess['A1'],
            sess['om0'], sess['om1'], sess['paired0'], sess['paired1'],
            template, fs)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        X_list.append(X)
        y_list.extend([label] * len(X))
        sid_list.extend([si] * len(X))
    
    X_all = np.vstack(X_list)
    y_all = np.array(y_list)
    sid_all = np.array(sid_list)
    
    # Per-session z-scoring
    X_z = np.zeros_like(X_all)
    for sid in np.unique(sid_all):
        mask = sid_all == sid
        mu = np.mean(X_all[mask], axis=0)
        sigma = np.std(X_all[mask], axis=0) + 1e-8
        X_z[mask] = (X_all[mask] - mu) / sigma
    
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_z)
    
    le = LabelEncoder()
    y_enc = le.fit_transform(y_all)
    
    clf = GradientBoostingClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
    clf.fit(X_s, y_enc)
    
    if verbose:
        print(f'Trained on {X_all.shape[0]} cycles x {X_all.shape[1]} features')
        print(f'Classes: {list(le.classes_)}')
    
    return {
        'clf': clf, 'scaler': scaler, 'label_encoder': le,
        'template': template, 'class_templates': class_templates,
        'feature_names': ALL_FEAT_NAMES,
        'X_train': X_all, 'y_train': y_all, 'session_ids': sid_all,
    }


def classify_session_v04(path_d0, path_d1, model, verbose=True):
    """
    Classify a session with transition rejection (confidence + template gating).
    """
    fs = CONFIG['FS']
    sess = process_session(path_d0, path_d1)
    if verbose:
        print(f'  DEBUG: {len(sess["paired0"])} paired cycles')
    
    X = extract_all_features(
        sess['t0'], sess['t1'], sess['A0'], sess['A1'],
        sess['om0'], sess['om1'], sess['paired0'], sess['paired1'],
        model['template'], fs)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Per-session z-scoring
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0) + 1e-8
    X_z = (X - mu) / sigma
    X_s = model['scaler'].transform(X_z)
    
    le = model['label_encoder']
    y_pred_enc = model['clf'].predict(X_s)
    y_pred_proba = model['clf'].predict_proba(X_s)
    pred_labels = le.inverse_transform(y_pred_enc)
    confidences = np.max(y_pred_proba, axis=1)
    
    # ── Transition rejection ──
    # Gate 1: confidence threshold
    conf_gate = confidences >= CONFIG['CONFIDENCE_THRESHOLD']
    
    # Gate 2: template correlation — cycle must match at least one class template
    tmpl_gate = np.zeros(len(X), dtype=bool)
    for ci in range(len(X)):
        cm = build_cycle_matrix(
            sess['A0'], sess['A1'], sess['om0'], sess['om1'],
            sess['paired0'][ci][0], sess['paired0'][ci][1],
            sess['paired1'][ci][0], sess['paired1'][ci][1],
            CONFIG['TARGET_LEN'])
        for cls_label, cls_tmpl in model['class_templates'].items():
            corr_mean, _ = template_correlation(cm, cls_tmpl)
            if corr_mean >= CONFIG['TEMPLATE_GATE_THRESHOLD']:
                tmpl_gate[ci] = True
                break
    
    # Apply gates
    labels = []
    for i in range(len(X)):
        if conf_gate[i] and tmpl_gate[i]:
            labels.append(pred_labels[i])
        else:
            labels.append('transition')
    
    # Session-level vote (exclude transitions)
    non_transition = [l for l in labels if l != 'transition']
    if non_transition:
        vote = Counter(non_transition)
        verdict = vote.most_common(1)[0][0]
    else:
        vote = Counter(labels)
        verdict = 'unknown'
    
    n_transition = sum(1 for l in labels if l == 'transition')
    
    if verbose:
        print(f'  {len(labels)} cycles | ', end='')
        for cls, cnt in sorted(Counter(labels).items()):
            print(f'{cls}={cnt} ', end='')
        print(f'-> {verdict}')
    
    return {
        'labels': labels, 'confidences': confidences.tolist(),
        'X': X, 'verdict': verdict, 'vote': dict(Counter(non_transition)),
        'n_transition': n_transition, 'session': sess,
    }


# ══════════════════════════════════════════════════════════════
# LOSO cross-validation with singleton handling
# ══════════════════════════════════════════════════════════════
def run_loso_v04(sessions, verbose=True):
    """
    Leave-one-session-out CV with singleton-class handling.
    Singletons are included in training but excluded from test folds.
    Reports both fine-grained and macro-class metrics.
    """
    # Count sessions per class
    class_counts = Counter(s[2] for s in sessions)
    singleton_classes = {c for c, n in class_counts.items() if n == 1}
    loso_sessions = [s for s in sessions if s[2] not in singleton_classes]
    
    if verbose and singleton_classes:
        print(f'Singleton classes (train-only): {sorted(singleton_classes)}')
        print(f'LOSO on {len(loso_sessions)} sessions ({len(sessions)-len(loso_sessions)} singletons excluded from test)')
        print('=' * 70)
    
    all_y_true, all_y_pred = [], []
    session_results = []
    
    for test_idx in range(len(loso_sessions)):
        test_d0, test_d1, test_label, test_name = loso_sessions[test_idx]
        
        # Train on everything except the test session
        train_data = []
        for i, s in enumerate(loso_sessions):
            if i != test_idx:
                train_data.append((s[0], s[1], s[2]))
        # Always include singletons in training
        for s in sessions:
            if s[2] in singleton_classes:
                train_data.append((s[0], s[1], s[2]))
        
        model = train_model_v04(train_data, verbose=False)
        result = classify_session_v04(test_d0, test_d1, model, verbose=False)
        
        if not result['labels']:
            if verbose:
                print(f'{test_name:<30s} NO CYCLES — skipped')
            continue
        correct = result['verdict'] == test_label
        session_results.append((test_name, test_label, result['verdict'], 
                                result['vote'], correct))
        
        # Per-cycle ground truth
        non_trans_labels = [l for l in result['labels'] if l != 'transition']
        all_y_true.extend([test_label] * len(non_trans_labels))
        all_y_pred.extend(non_trans_labels)
        
        if verbose:
            status = 'CORRECT' if correct else 'WRONG'
            print(f'{test_name:<30s} GT={test_label:<18s} -> {result["verdict"]:<18s} [{status}]')
    
    n_correct = sum(1 for r in session_results if r[4])
    session_acc = n_correct / len(session_results) if session_results else 0
    
    # Fine-grained metrics
    le = LabelEncoder()
    all_classes = sorted(set(all_y_true + all_y_pred))
    le.fit(all_classes)
    
    result_dict = {
        'session_results': session_results,
        'n_correct': n_correct, 'n_total': len(session_results),
        'session_acc': session_acc,
    }
    
    if all_y_true:
        y_t = le.transform(all_y_true)
        y_p = le.transform(all_y_pred)
        result_dict['fine_report'] = classification_report(
            y_t, y_p, target_names=le.classes_, zero_division=0)
        result_dict['fine_f1_macro'] = f1_score(y_t, y_p, average='macro', zero_division=0)
        result_dict['fine_f1_weighted'] = f1_score(y_t, y_p, average='weighted', zero_division=0)
        result_dict['fine_cm'] = confusion_matrix(y_t, y_p)
        result_dict['fine_classes'] = le.classes_
        result_dict['all_y_true'] = all_y_true
        result_dict['all_y_pred'] = all_y_pred
        
        # Macro-class
        macro_true = [MACRO_MAP.get(l, 'unknown') for l in all_y_true]
        macro_pred = [MACRO_MAP.get(l, 'unknown') for l in all_y_pred]
        le_m = LabelEncoder()
        all_macro = sorted(set(macro_true + macro_pred))
        le_m.fit(all_macro)
        mt = le_m.transform(macro_true)
        mp = le_m.transform(macro_pred)
        result_dict['macro_report'] = classification_report(
            mt, mp, target_names=le_m.classes_, zero_division=0)
        result_dict['macro_f1_macro'] = f1_score(mt, mp, average='macro', zero_division=0)
        result_dict['macro_cm'] = confusion_matrix(mt, mp)
        result_dict['macro_classes'] = le_m.classes_
    
    return result_dict


# ══════════════════════════════════════════════════════════════
# Report figure generators
# ══════════════════════════════════════════════════════════════
def plot_loso_confusion(results, level='fine', save_path=None):
    """Plot confusion matrix from LOSO results."""
    key_cm = f'{level}_cm'
    key_cls = f'{level}_classes'
    if key_cm not in results:
        print(f'No {level} confusion matrix available')
        return
    cm = results[key_cm]
    classes = results[key_cls]
    
    # Normalize
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    
    fig, ax = plt.subplots(figsize=(max(5, len(classes)), max(4, len(classes)-1)))
    disp = ConfusionMatrixDisplay(cm_norm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2f')
    title = f'LOSO confusion matrix ({level}-grained, normalized)'
    ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


def plot_classification_timeline(result, title='', save_path=None):
    """Plot per-cycle classification as colored bars with confidence overlay."""
    labels = result['labels']
    confs = result['confidences']
    n = len(labels)
    
    fig, ax = plt.subplots(figsize=(14, 3))
    for i in range(n):
        color = PAL.get(labels[i], '#888888')
        ax.bar(i, 1, color=color, alpha=0.7, width=1.0)
    
    ax2 = ax.twinx()
    ax2.plot(range(n), confs, 'k-', lw=0.8, alpha=0.6)
    ax2.axhline(CONFIG['CONFIDENCE_THRESHOLD'], color='red', ls='--', lw=0.8, alpha=0.5)
    ax2.set_ylabel('Confidence')
    ax2.set_ylim(0, 1)
    
    ax.set_xlabel('Cycle index')
    ax.set_yticks([])
    ax.set_title(f'{title} — verdict: {result["verdict"]}')
    
    # Legend
    from matplotlib.patches import Patch
    used = sorted(set(labels))
    handles = [Patch(facecolor=PAL.get(l, '#888'), label=l, alpha=0.7) for l in used]
    ax.legend(handles=handles, fontsize=7, loc='upper right', ncol=min(4, len(used)))
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

---
## 3. Auto-discover sessions

In [12]:
SESSIONS = discover_sessions(DATA_PROCESSED)
by_class = summarize_sessions(SESSIONS)

Discovered 13 labeled sessions across 7 classes:
  cheetahs_tail       : 1 sessions [SINGLETON]
    - cheetahs_tail_jo
  dragon_roll         : 2 sessions [LOSO OK]
    - dragon_roll_oli
    - dragon_roll_jo
  overhand            : 2 sessions [LOSO OK]
    - overhand_oli
    - overhand_jo
  race_and_chase      : 1 sessions [SINGLETON]
    - race_and_chase_jo
  sneak_overhand      : 2 sessions [LOSO OK]
    - sneak_overhand_oli
    - sneak_overhand_jo
  sneak_underhand     : 1 sessions [SINGLETON]
    - sneak_underhand_jo
  underhand           : 4 sessions [LOSO OK]
    - underhand_default
    - underhand_default
    - underhand_jo
    - underhand_oli



---
## 4. LOSO cross-validation (7-class with transition rejection)

In [13]:
loso_results = run_loso_v04(SESSIONS, verbose=True)

print(f'\n{"="*70}')
print(f'Session accuracy: {loso_results["n_correct"]}/{loso_results["n_total"]} '
      f'({loso_results["session_acc"]*100:.0f}%)')

if 'fine_report' in loso_results:
    print(f'\n--- Fine-grained per-cycle ---')
    print(loso_results['fine_report'])
    print(f'F1 macro={loso_results["fine_f1_macro"]:.3f}')

if 'macro_report' in loso_results:
    print(f'\n--- Macro-class per-cycle ---')
    print(f'(foundational=overhand+underhand, sneak=sneak_*, advanced=dragon_roll+race_and_chase+cheetahs_tail)')
    print(loso_results['macro_report'])
    print(f'F1 macro={loso_results["macro_f1_macro"]:.3f}')

Singleton classes (train-only): ['cheetahs_tail', 'race_and_chase', 'sneak_underhand']
LOSO on 10 sessions (3 singletons excluded from test)


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

---
## 5. Full model (all 13 sessions)

In [ ]:
all_data = [(s[0], s[1], s[2]) for s in SESSIONS]
model_full = train_model_v04(all_data, verbose=True)

print(f'\nSelf-test:')
for d0, d1, label, name in SESSIONS:
    result = classify_session_v04(d0, d1, model_full, verbose=False)
    status = 'CORRECT' if result['verdict'] == label else 'WRONG'
    trans = result['n_transition']
    print(f'  {name:<30s} GT={label:<18s} -> {result["verdict"]:<18s} '
          f'(trans={trans}) [{status}]')

---
## 6. Report figures

### 6.1 LOSO confusion matrices

In [ ]:
plot_loso_confusion(loso_results, level='fine', save_path='fig_cm_fine.png')
plot_loso_confusion(loso_results, level='macro', save_path='fig_cm_macro.png')

### 6.2 Feature importance (tier-colored)

In [ ]:
imp = model_full['clf'].feature_importances_
feat_names_all = model_full['feature_names']

def get_tier(feat_name):
    idx = feat_names_all.index(feat_name)
    if idx < 12: return 'Physics'
    elif idx < 28: return 'Fourier'
    elif idx < 36: return 'SVD'
    elif idx < 38: return 'Template'
    elif idx < 47: return 'DMD'
    else: return 'Subject-inv.'

tier_colors = {
    'Physics': '#4CAF50', 'Fourier': '#2196F3', 'SVD': '#FF9800',
    'Template': '#9C27B0', 'DMD': '#F44336', 'Subject-inv.': '#795548'
}

top15_idx = np.argsort(imp)[::-1][:15]
top15_names = [feat_names_all[i] for i in top15_idx]
top15_imp = imp[top15_idx]
top15_tiers = [get_tier(n) for n in top15_names]
top15_colors = [tier_colors[t] for t in top15_tiers]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(range(len(top15_names)), top15_imp[::-1], color=top15_colors[::-1], alpha=0.85)
ax.set_yticks(range(len(top15_names)))
ax.set_yticklabels(top15_names[::-1], fontsize=9)
ax.set_xlabel('GBM importance')
ax.set_title('Top 15 features (7-class)')

from matplotlib.patches import Patch
used_tiers = list(dict.fromkeys(top15_tiers))
legend_handles = [Patch(facecolor=tier_colors[t], label=t, alpha=0.85) for t in used_tiers]
ax.legend(handles=legend_handles, fontsize=8, loc='lower right', title='Feature tier')
plt.tight_layout()
plt.savefig('fig_feature_importance_7class.png', dpi=200, bbox_inches='tight')
plt.show()

# Tier summary
print('Feature tier importance:')
tier_ranges = {'Physics(12D)':(0,12), 'Fourier(16D)':(12,28), 'SVD(8D)':(28,36),
               'Template(2D)':(36,38), 'DMD(9D)':(38,47), 'SubjInv(9D)':(47,56)}
for name, (lo,hi) in tier_ranges.items():
    t = np.sum(imp[lo:hi])
    print(f'  {name:<20s} {t:.3f} ({t/np.sum(imp)*100:.1f}%)')

### 6.3 Classification timeline (example session)

In [ ]:
# Show timeline for first session
d0, d1, label, name = SESSIONS[0]
result_ex = classify_session_v04(d0, d1, model_full, verbose=False)
plot_classification_timeline(result_ex, title=name, save_path='fig_timeline_example.png')

### 6.4 SVD spectrum (overhand)

In [ ]:
# SVD spectrum across multiple cycles from first overhand session
oh_sessions = [s for s in SESSIONS if s[2] == 'overhand']
if oh_sessions:
    sess_oh = process_session(oh_sessions[0][0], oh_sessions[0][1])
    cms_oh = sess_oh['cycle_matrices']
    
    n_show = min(20, len(cms_oh))
    sample_idx = np.linspace(0, len(cms_oh)-1, n_show, dtype=int)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
    
    S_all = []
    for idx in sample_idx:
        _, S_s, _ = svd(cms_oh[idx], full_matrices=False)
        S_norm = S_s / S_s[0]
        S_all.append(S_norm)
        ax1.plot(range(len(S_norm)), S_norm, 'o-', color='#7F77DD', alpha=0.15, ms=3, lw=0.8)
    
    S_mean = np.mean(S_all, axis=0)
    ax1.plot(range(len(S_mean)), S_mean, 'o-', color='#E24B4A', lw=2, ms=5,
             label=f'mean (n={len(cms_oh)})', zorder=10)
    ax1.set_xlabel('Mode index'); ax1.set_ylabel('sigma_i / sigma_0')
    ax1.set_title('Normalized singular value spectrum'); ax1.legend(fontsize=8)
    
    cumvar_all = []
    for cm in cms_oh:
        _, S_s, _ = svd(cm, full_matrices=False)
        cv = np.cumsum(S_s**2) / np.sum(S_s**2) * 100
        cumvar_all.append(cv)
        ax2.plot(range(1, len(cv)+1), cv, '-', color='#7F77DD', alpha=0.1, lw=0.8)
    
    cv_mean = np.mean(cumvar_all, axis=0)
    ax2.plot(range(1, len(cv_mean)+1), cv_mean, 'o-', color='#E24B4A', lw=2, ms=5, label='mean')
    ax2.axhline(99, color='gray', ls='--', lw=1, label='99%')
    r99 = int(np.searchsorted(cv_mean, 99)) + 1
    ax2.axvline(r99, color='#E24B4A', ls='--', lw=1, alpha=0.5)
    ax2.annotate(f'r={r99}', xy=(r99, 99), xytext=(r99+0.5, 92), fontsize=9, color='#E24B4A')
    ax2.set_xlabel('Number of modes'); ax2.set_ylabel('Cumulative variance [%]')
    ax2.set_title('Low-rank structure'); ax2.legend(fontsize=8)
    
    plt.suptitle('SVD analysis - overhand cycle matrix', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig('fig_svd_spectrum.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f'99% variance at r={r99} modes')